In [1]:
#import library
import mysql.connector
import pandas as pd
import numpy as np
import random
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Koneksi ke database MySQL
def fetch_questions_from_db():
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="1234",
        database="skripsi"
    )
    
    cursor = conn.cursor(dictionary=True)
    cursor.execute("SELECT id, kategori, sub_kategori, teks_soal FROM soal")
    questions = cursor.fetchall()
    
    conn.close()
    return questions


In [3]:
# Menghitung cosine similarity antara soal-soal
def select_questions_by_category_and_subcategory(questions, total_questions):
    selected_questions = []
    categories = list(set([q['kategori'] for q in questions]))
    subcategories = list(set([q['sub_kategori'] for q in questions]))
    
    num_per_category = total_questions // len(categories)
    num_per_subcategory = total_questions // len(subcategories)
    
    for category in categories:
        category_questions = [q for q in questions if q['kategori'] == category]
        for subcategory in subcategories:
            subcategory_questions = [q for q in category_questions if q['sub_kategori'] == subcategory]
            selected_questions.extend(random.sample(subcategory_questions, min(num_per_subcategory, len(subcategory_questions))))
    
    return selected_questions[:total_questions]



In [4]:
# Mengambil soal secara merata berdasarkan kategori dan sub-kategori
def calculate_cosine_similarity(questions):
    texts = [q['teks_soal'] for q in questions]
    tfidf_matrix = np.array([[len(word) for word in text.split()] for text in texts])
    
    return cosine_similarity(tfidf_matrix)


In [5]:
# Menampilkan hasil pengacakan dalam bentuk DataFrame
def shuffle_questions(questions, cosine_sim):
    shuffled_questions = []
    
    for i, question in enumerate(questions):
        sim_scores = list(enumerate(cosine_sim[i]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        similar_questions = [questions[idx] for idx, score in sim_scores if idx != i]
        
        random.shuffle(similar_questions)
        
        # Tidak mengubah id, kategori, sub_kategori, dan teks_soal
        shuffled_questions.append({
            "id": question['id'],
            "kategori": question['kategori'],
            "sub_kategori": question['sub_kategori'],
            "teks_soal": question['teks_soal']
        })
    
    return shuffled_questions


In [6]:
def calculate_distribution(shuffled_questions):
    # Menghitung distribusi berdasarkan kategori
    category_distribution = pd.Series([q['kategori'] for q in shuffled_questions]).value_counts()
    
    # Menghitung distribusi berdasarkan sub-kategori
    subcategory_distribution = pd.Series([q['sub_kategori'] for q in shuffled_questions]).value_counts()
    
    return category_distribution, subcategory_distribution



In [7]:
def calculate_accuracy_distribution(shuffled_questions, total_questions, categories, subcategories):
    # Menghitung distribusi ideal
    ideal_per_category = total_questions / len(categories)
    ideal_per_subcategory = total_questions / len(subcategories)
    
    # Menghitung distribusi aktual
    category_distribution = pd.Series([q['kategori'] for q in shuffled_questions]).value_counts()
    subcategory_distribution = pd.Series([q['sub_kategori'] for q in shuffled_questions]).value_counts()
    
    # Menghitung akurasi untuk kategori
    category_accuracy = (category_distribution / ideal_per_category).mean() * 100
    
    # Menghitung akurasi untuk sub-kategori
    subcategory_accuracy = (subcategory_distribution / ideal_per_subcategory).mean() * 100
    
    return category_accuracy, subcategory_accuracy


In [8]:
def display_shuffled_questions_as_text(shuffled_questions):
    for i, question in enumerate(shuffled_questions, start=1):
        print(f"Soal {i}:")
        print(f"ID: {question['id']}")
        print(f"Kategori: {question['kategori']}")
        print(f"Sub-Kategori: {question['sub_kategori']}")
        print(f"Teks Soal: {question['teks_soal']}")
        print("-" * 40)


In [9]:
# Main function
questions = fetch_questions_from_db()
categories = list(set([q['kategori'] for q in questions]))
subcategories = list(set([q['sub_kategori'] for q in questions]))

selected_questions = select_questions_by_category_and_subcategory(questions, total_questions=50)
cosine_sim = calculate_cosine_similarity(selected_questions)
shuffled_questions = shuffle_questions(selected_questions, cosine_sim)

# Menghitung dan menampilkan akurasi distribusi
category_accuracy, subcategory_accuracy = calculate_accuracy_distribution(shuffled_questions, 50, categories, subcategories)

print(f"Akurasi Distribusi berdasarkan Kategori: {category_accuracy:.2f}%")
print(f"Akurasi Distribusi berdasarkan Sub-Kategori: {subcategory_accuracy:.2f}%")

# Menampilkan hasil distribusi
category_distribution, subcategory_distribution = calculate_distribution(shuffled_questions)
print("\nDistribusi berdasarkan Kategori:")
print(category_distribution)
print("\nDistribusi berdasarkan Sub-Kategori:")
print(subcategory_distribution)

# Menampilkan soal yang telah diacak dalam format teks
display_shuffled_questions_as_text(shuffled_questions)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.